# Heart Disease Prediction
Kaggle Playground (Heart Disease) pipeline: load `train.csv` / `test.csv`, engineer features, compare models with cross-validation, tune the best ones, evaluate on a hold-out set, then refit on all labelled data and write `submission.csv`.

**Files expected next to this notebook:** `train.csv`, `test.csv`, `sample_submission.csv`.

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings('ignore', category=UserWarning)
%matplotlib inline

SEED = 42
SAMPLE_N = 30_000
CV_FOLDS = 5
TUNE_FOLDS = 3
SUBMIT_PROBA = True
DATA_DIRS = ['.', 'data', '/kaggle/input/playground-series-s6e2', '/mnt/user-data/uploads']

## 1. Load and Explore Data

In [2]:
DATA_DIR = next((d for d in DATA_DIRS if os.path.exists(os.path.join(d, 'train.csv'))), None)
if DATA_DIR is None:
    raise FileNotFoundError("train.csv not found. Put train.csv, test.csv and sample_submission.csv next to this notebook.")

train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

ID_COL, TARGET_COL = sample_sub.columns[0], sample_sub.columns[1]

y = train[TARGET_COL]
if not pd.api.types.is_numeric_dtype(y):
    y = (y.astype(str).str.strip().str.lower()
          .map({'presence': 1, 'absence': 0, 'yes': 1, 'no': 0, '1': 1, '0': 0}))
if y.isna().any():
    raise ValueError(f"Unrecognised values in target column: {train[TARGET_COL].unique()}")
y = y.astype(int)

print(f"Train: {train.shape} | Test: {test.shape} | Sample submission: {sample_sub.shape}")
print(f"ID column: {ID_COL!r} | Target column: {TARGET_COL!r}")
print(f"\nMissing values in train: {int(train.isnull().sum().sum())}")
print(f"\nTarget distribution:\n{y.value_counts(normalize=True).round(4)}")

Train: (630000, 15) | Test: (270000, 14) | Sample submission: (270000, 2)
ID column: 'id' | Target column: 'Heart Disease'

Missing values in train: 0

Target distribution:
Heart Disease
0    0.5517
1    0.4483
Name: proportion, dtype: float64


In [3]:
train.info()
train.head()

<class 'pandas.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       630000 non-null  int64  
 1   Age                      630000 non-null  int64  
 2   Sex                      630000 non-null  int64  
 3   Chest pain type          630000 non-null  int64  
 4   BP                       630000 non-null  int64  
 5   Cholesterol              630000 non-null  int64  
 6   FBS over 120             630000 non-null  int64  
 7   EKG results              630000 non-null  int64  
 8   Max HR                   630000 non-null  int64  
 9   Exercise angina          630000 non-null  int64  
 10  ST depression            630000 non-null  float64
 11  Slope of ST              630000 non-null  int64  
 12  Number of vessels fluro  630000 non-null  int64  
 13  Thallium                 630000 non-null  int64  
 14  Heart Disease  

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,Absence
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,Presence


## 2. Feature Engineering
Column names are mapped to the classic UCI names (works with both the Playground names such as `Max HR` and the original `thalach`). All transformations are row-wise, so the same function is applied identically to train and test.

In [4]:
def _norm(s):
    return ''.join(ch for ch in s.lower() if ch.isalnum())

RENAME = {
    'age': 'age', 'sex': 'sex',
    'chestpaintype': 'cp', 'cp': 'cp',
    'bp': 'trestbps', 'trestbps': 'trestbps',
    'cholesterol': 'chol', 'chol': 'chol',
    'fbsover120': 'fbs', 'fbs': 'fbs',
    'ekgresults': 'restecg', 'restecg': 'restecg',
    'maxhr': 'thalach', 'thalach': 'thalach',
    'exerciseangina': 'exang', 'exang': 'exang',
    'stdepression': 'oldpeak', 'oldpeak': 'oldpeak',
    'slopeofst': 'slope', 'slope': 'slope',
    'numberofvesselsfluro': 'ca', 'ca': 'ca',
    'thallium': 'thal', 'thal': 'thal',
}
BASE = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
        'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']

def standardise(df):
    out = df.drop(columns=[c for c in (ID_COL, TARGET_COL) if c in df.columns])
    out = out.rename(columns={c: RENAME[_norm(c)] for c in out.columns if _norm(c) in RENAME})
    missing = [c for c in BASE if c not in out.columns]
    if missing:
        raise KeyError(f"Missing expected columns {missing}. Found: {list(df.columns)}")
    return out[BASE].astype(float)

def add_features(df):
    f = df.copy()
    age = f['age'].clip(lower=1)
    max_hr = (220 - f['age']).clip(lower=1)

    f['chol_per_age'] = f['chol'] / age
    f['hr_pct_of_max'] = f['thalach'] / max_hr
    f['hr_deficit'] = max_hr - f['thalach']
    f['bp_chol_ratio'] = f['trestbps'] / f['chol'].replace(0, np.nan)
    f['age_x_oldpeak'] = f['age'] * f['oldpeak']
    f['oldpeak_x_slope'] = f['oldpeak'] * f['slope']
    f['age_bin'] = np.digitize(f['age'], [40, 50, 60])
    f['any_vessels'] = (f['ca'] > 0).astype(int)
    f['has_st_depression'] = (f['oldpeak'] > 0).astype(int)

    return f.replace([np.inf, -np.inf], np.nan)

X = add_features(standardise(train))
X_test = add_features(standardise(test))[X.columns]
feature_names = X.columns.tolist()
print(f"Base features: {len(BASE)} | Engineered: {len(feature_names) - len(BASE)} | Total: {len(feature_names)}")

Base features: 13 | Engineered: 9 | Total: 22


## 3. Train / Hold-out Split

In [5]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f"Train: {X_train.shape} | Hold-out: {X_valid.shape}")

if len(X_train) > SAMPLE_N:
    X_s, _, y_s, _ = train_test_split(X_train, y_train, train_size=SAMPLE_N,
                                      random_state=SEED, stratify=y_train)
else:
    X_s, y_s = X_train, y_train
print(f"Sample used for CV / tuning: {X_s.shape}")

Train: (504000, 22) | Hold-out: (126000, 22)
Sample used for CV / tuning: (30000, 22)


## 4. Model Selection
Every model lives in a `Pipeline` (median imputation, plus `StandardScaler` for the scale-sensitive models: Logistic Regression, SVM, Naive Bayes). Scaling is fitted inside each CV fold, so there is no leakage. Models are scored with ROC-AUC.

In [6]:
def make_pipe(clf, scale=False):
    steps = [('imp', SimpleImputer(strategy='median'))]
    if scale:
        steps.append(('scale', StandardScaler()))
    steps.append(('clf', clf))
    return Pipeline(steps)

models = {
    'Logistic Regression': make_pipe(LogisticRegression(max_iter=2000, random_state=SEED), scale=True),
    'Random Forest':       make_pipe(RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED)),
    'Gradient Boosting':   make_pipe(GradientBoostingClassifier(n_estimators=150, random_state=SEED)),
    'SVM':                 make_pipe(SVC(kernel='rbf', random_state=SEED), scale=True),
    'Naive Bayes':         make_pipe(GaussianNB(), scale=True),
    'XGBoost':             make_pipe(xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                                                       eval_metric='logloss', n_jobs=-1, random_state=SEED)),
    'LightGBM':            make_pipe(lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05,
                                                        random_state=SEED, verbose=-1)),
}

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
cv_results = []
for name, model in models.items():
    scores = cross_val_score(model, X_s, y_s, cv=cv, scoring='roc_auc')
    cv_results.append({'model': name, 'mean_auc': scores.mean(), 'std': scores.std()})
    print(f"{name:20s} AUC = {scores.mean():.4f} +/- {scores.std():.4f}")

results_df = pd.DataFrame(cv_results).sort_values('mean_auc', ascending=False).reset_index(drop=True)
results_df

Logistic Regression  AUC = 0.9515 +/- 0.0022
Random Forest        AUC = 0.9475 +/- 0.0020
Gradient Boosting    AUC = 0.9537 +/- 0.0019
SVM                  AUC = 0.9442 +/- 0.0027
Naive Bayes          AUC = 0.9286 +/- 0.0027
XGBoost              AUC = 0.9536 +/- 0.0020
LightGBM             AUC = 0.9521 +/- 0.0022


,model,mean_auc,std
0,Gradient Boosting,0.953699,0.001861
1,XGBoost,0.953601,0.002003
2,LightGBM,0.952097,0.002232
3,Logistic Regression,0.951458,0.002163
4,Random Forest,0.947470,0.001987
5,SVM,0.944195,0.002694
6,Naive Bayes,0.928633,0.002670


In [7]:
plt.figure(figsize=(10, 5))
plt.barh(results_df['model'][::-1], results_df['mean_auc'][::-1], xerr=results_df['std'][::-1])
lo = max(0.0, results_df['mean_auc'].min() - 0.05)
plt.xlim(lo, min(1.0, results_df['mean_auc'].max() + 0.02))
plt.xlabel('ROC-AUC (cross-validated)')
plt.title('Model Comparison')
plt.tight_layout()
plt.show()

## 5. Model Optimization

In [8]:
param_grids = {
    'Logistic Regression': {'clf__C': [0.1, 1, 10]},
    'Random Forest':       {'clf__max_depth': [6, 10, None], 'clf__min_samples_leaf': [1, 5]},
    'Gradient Boosting':   {'clf__max_depth': [2, 3, 4], 'clf__learning_rate': [0.05, 0.1]},
    'SVM':                 {'clf__C': [0.5, 1, 5]},
    'Naive Bayes':         {'clf__var_smoothing': [1e-9, 1e-8, 1e-7]},
    'XGBoost':             {'clf__max_depth': [3, 4, 6], 'clf__learning_rate': [0.03, 0.1],
                            'clf__n_estimators': [200, 400]},
    'LightGBM':            {'clf__num_leaves': [15, 31, 63], 'clf__learning_rate': [0.03, 0.1],
                            'clf__n_estimators': [200, 400]},
}

top_models = results_df.head(3)['model'].tolist()
print(f"Top models to optimize: {top_models}")

tune_cv = StratifiedKFold(n_splits=TUNE_FOLDS, shuffle=True, random_state=SEED)
best_name, best_score, best_params = None, -np.inf, None
optimized_results = []

for name in top_models:
    grid = GridSearchCV(models[name], param_grids[name], cv=tune_cv, scoring='roc_auc', n_jobs=1)
    grid.fit(X_s, y_s)
    optimized_results.append({'model': name, 'best_auc': grid.best_score_, **grid.best_params_})
    print(f"{name:20s} best AUC = {grid.best_score_:.4f}  {grid.best_params_}")
    if grid.best_score_ > best_score:
        best_name, best_score, best_params = name, grid.best_score_, grid.best_params_

pd.DataFrame(optimized_results)

Top models to optimize: ['Gradient Boosting', 'XGBoost', 'LightGBM']
Gradient Boosting    best AUC = 0.9539  {'clf__learning_rate': 0.1, 'clf__max_depth': 2}
XGBoost              best AUC = 0.9542  {'clf__learning_rate': 0.03, 'clf__max_depth': 3, 'clf__n_estimators': 400}
LightGBM             best AUC = 0.9536  {'clf__learning_rate': 0.03, 'clf__n_estimators': 200, 'clf__num_leaves': 15}


,model,best_auc,clf__learning_rate,clf__max_depth,clf__n_estimators,clf__num_leaves
0,Gradient Boosting,0.953869,0.10,2.0,NaN,NaN
1,XGBoost,0.954190,0.03,3.0,400.0,NaN
2,LightGBM,0.953640,0.03,NaN,200.0,15.0


## 6. Final Evaluation on the Hold-out Set

In [9]:
best_model = clone(models[best_name]).set_params(**best_params)
best_model.fit(X_train, y_train)

print(f"Best model: {best_name}")
print(f"Best tuned CV AUC (on sample): {best_score:.4f}")

proba = best_model.predict_proba(X_valid)[:, 1]
pred = (proba >= 0.5).astype(int)

print("\nHold-out performance:")
print(f"Accuracy:  {accuracy_score(y_valid, pred):.4f}")
print(f"F1-Score:  {f1_score(y_valid, pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_valid, proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_valid, pred))

Best model: XGBoost
Best tuned CV AUC (on sample): 0.9542

Hold-out performance:
Accuracy:  0.8880
F1-Score:  0.8739
ROC-AUC:   0.9549

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.91      0.90     69509
           1       0.88      0.87      0.87     56491

    accuracy                           0.89    126000
   macro avg       0.89      0.89      0.89    126000
weighted avg       0.89      0.89      0.89    126000



## 7. Feature Importance

In [10]:
clf = best_model.named_steps['clf']
if hasattr(clf, 'feature_importances_'):
    importance = clf.feature_importances_
elif hasattr(clf, 'coef_'):
    importance = np.abs(clf.coef_).ravel()
else:
    importance = None
    print(f"{best_name} has no built-in feature importance.")

if importance is not None:
    top = np.argsort(importance)[::-1][:15]
    plt.figure(figsize=(10, 7))
    plt.barh(range(len(top))[::-1], importance[top])
    plt.yticks(range(len(top))[::-1], [feature_names[i] for i in top])
    plt.xlabel('Feature Importance')
    plt.title(f'Top {len(top)} Features ({best_name})')
    plt.tight_layout()
    plt.show()

## 8. Create `submission.csv`
The final model is refitted on **all** labelled rows (train + hold-out) before predicting the test set.

In [11]:
final_model = clone(models[best_name]).set_params(**best_params)
final_model.fit(X, y)

test_proba = final_model.predict_proba(X_test)[:, 1]
preds = test_proba if SUBMIT_PROBA else (test_proba >= 0.5).astype(int)

submission = pd.DataFrame({ID_COL: test[ID_COL].values, TARGET_COL: preds})
submission.to_csv('submission.csv', index=False)

assert list(submission.columns) == list(sample_sub.columns), "Column names differ from sample_submission"
assert len(submission) == len(sample_sub), "Row count differs from sample_submission"
assert (submission[ID_COL].values == sample_sub[ID_COL].values).all(), "IDs differ / are in a different order"
assert submission[TARGET_COL].notna().all(), "NaN predictions"
print(f"Saved submission.csv  shape={submission.shape}")
submission.head()

Saved submission.csv  shape=(270000, 2)


,id,Heart Disease
0,630000,0.871379
1,630001,0.009005
2,630002,0.981260
3,630003,0.005977
4,630004,0.310497


## Summary
- **Data**: loads `train.csv`/`test.csv`, auto-detects id/target columns from `sample_submission.csv`, and handles a text target (`Presence`/`Absence`).
- **Features**: 9 engineered features (ratios, age-predicted max heart-rate features, interactions, age band, indicators) built by one function applied identically to train and test.
- **Model selection**: 7 classifiers in leak-free pipelines, 5-fold CV, scored by ROC-AUC.
- **Tuning**: `GridSearchCV` on the top 3 models; the best configuration is evaluated on a hold-out set, then refitted on all labelled data for the test predictions.
- **Output**: `submission.csv`, validated against `sample_submission.csv`. Set `SUBMIT_PROBA = False` if the competition metric needs hard 0/1 labels.